# Model 3 — Damage Tier & Depreciation Analysis (K-Means Clustering)

**Question:** Which damage severity tier does this car belong to — and how much does each tier depreciate a car's resale value compared to a pristine car?

---

## Introduction

This notebook uses K-Means clustering to group cars into damage severity tiers based on their body condition features, then analyses how much each tier depreciates resale value relative to a pristine baseline.

**Key information:**

- **Dataset:** Uses **unscaled** data. Do not use the scaled dataset.
- **Clustering scope:** Clusters are built on damage-related columns only. The `Fiyat` column is then joined for depreciation analysis.
- **Algorithm:** You **must** use `KMeans` only. No other clustering algorithm is permitted.
- **Recommended k:** 4 (Pristine / Light Wear / Moderate Damage / Heavy Damage). Use the Elbow Method below to justify your final choice.

**What you can freely change:**
- Adjust the damage features used for clustering.
- Apply feature weights (e.g., multiply higher-severity columns by a weight factor).
- Choose a different k based on the Elbow Method.
- Tune `n_estimators`, `n_init`, or other KMeans hyperparameters.

**What you cannot change:**
- The general technique category: this must remain K-Means clustering.

## 1. Data Import

Load the unscaled dataset and import all required libraries. **This cell is complete — do not modify it.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

df = pd.read_csv('../../data/proceed_dataset_without_scaling.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Feature Selection & Scaling

Select damage-related features and scale them with `StandardScaler` so that KMeans distances are not dominated by features with large numeric ranges.

**TODO:** You may add, remove, or replace features from the recommended set. You may also apply manual weights before scaling.

In [ ]:
# TODO: Select damage-related features for clustering.
# Recommended starting set (all panel/body part condition columns):
damage_features = [
    'total_painted_parts', 'total_changed_parts', 'paint_damage_score',
    'sığ_arka_çamurluk_durumu', 'sol_arka_çamurluk_durumu',
    'sığ_ön_çamurluk_durumu', 'sol_ön_çamurluk_durumu',
    'sığ_arka_kapı_durumu', 'sol_arka_kapı_durumu',
    'sığ_ön_kapı_durumu', 'sol_ön_kapı_durumu',
    'arka_kaput_durumu', 'motor_kaputu_durumu',
    'ön_tampon_durumu', 'arka_tampon_durumu', 'tavan_durumu'
]
features = [f for f in damage_features if f in df.columns]
X_damage = df[features].fillna(0)

# Scale features for KMeans
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_damage)
print(f"Clustering feature matrix shape: {X_scaled.shape}")

## 3. Elbow Method — Choose Optimal k

Run KMeans for k = 2 … 10 and plot inertia. Look for the "elbow" — the point where the rate of inertia decrease slows sharply. This is your recommended k.

> **⚠️ This cell computes and plots the Elbow Method. Run after loading data.**

In [ ]:
# ⚠️ This cell computes and plots the Elbow Method. Run after loading data.
inertias = []
k_range = range(2, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_range, inertias, 'bo-', markersize=8, linewidth=2)
ax.set_xlabel('Number of Clusters (k)', fontsize=13)
ax.set_ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=13)
ax.set_title('Elbow Method — Choosing Optimal k for K-Means', fontsize=15)
ax.set_xticks(list(k_range))
ax.axvline(x=4, color='red', linestyle='--', linewidth=1.5, label='Recommended k=4')
ax.legend()
plt.tight_layout()
plt.show()
print("Use the elbow point (where inertia decrease sharply slows down) to justify your choice of k.")

## 4. Model Training — K-Means Clustering

Fit KMeans with your chosen k and assign a cluster label to every row in the dataset.

**TODO:** Set `k` based on your Elbow Method analysis above. The recommended value is 4.

In [ ]:
# TODO: Set n_clusters based on your Elbow Method analysis. Recommended: 4.
k = 4  # TODO: adjust after reviewing the elbow plot

kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
kmeans.fit(X_scaled)
df['cluster'] = kmeans.labels_
print(f"Cluster distribution:\n{df['cluster'].value_counts().sort_index()}")

## Step: Manually Assign Cluster Labels

After running the clustering, inspect the average `paint_damage_score` per cluster (shown in the heatmap below). The cluster with the **lowest** average `paint_damage_score` is the baseline (**Pristine**). Assign human-readable labels in the cell below. Example mapping:

| Cluster | Label |
|---------|-------|
| 0 | Pristine |
| 1 | Light Wear |
| 2 | Moderate Damage |
| 3 | Heavy Damage |

Your actual mapping will depend on your clustering results.

## 5. Cluster Label Assignment

Map numeric cluster IDs to human-readable damage tier names.

**TODO:** After inspecting the heatmap (Section 9), update the mapping below to match your actual clusters. The cluster with the lowest average `paint_damage_score` must be labelled `'Pristine'`.

In [ ]:
# TODO: After inspecting the heatmap, update this mapping to match your clusters.
# The cluster with the lowest paint_damage_score should be labelled 'Pristine'.
cluster_labels = {
    0: 'Pristine',        # TODO: verify this matches lowest damage cluster
    1: 'Light Wear',      # TODO: adjust
    2: 'Moderate Damage', # TODO: adjust
    3: 'Heavy Damage'     # TODO: adjust
}
df['cluster_label'] = df['cluster'].map(cluster_labels)
print(df['cluster_label'].value_counts())

## 6. Evaluation — Depreciation Bar Chart

Plot the median price per damage tier and annotate each bar with the percentage change relative to the Pristine baseline. This is the core result of the model.

> **⚠️ Replace with your actual cluster results. Ensure `cluster_labels` is updated first.**

In [ ]:
# ⚠️ Replace with your actual cluster results. Ensure cluster_labels is updated first.
cluster_prices = df.groupby('cluster_label')['Fiyat'].median().reset_index()
cluster_prices.columns = ['Cluster', 'Median Price (TL)']

# Identify pristine cluster median as baseline
pristine_median = cluster_prices.loc[cluster_prices['Cluster'] == 'Pristine', 'Median Price (TL)'].values[0]
cluster_prices['Depreciation (%)'] = ((cluster_prices['Median Price (TL)'] - pristine_median) / pristine_median * 100).round(1)

order = ['Pristine', 'Light Wear', 'Moderate Damage', 'Heavy Damage']
order_existing = [o for o in order if o in cluster_prices['Cluster'].values]
cluster_prices = cluster_prices.set_index('Cluster').loc[order_existing].reset_index()

colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c'][:len(order_existing)]
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(cluster_prices['Cluster'], cluster_prices['Median Price (TL)'],
              color=colors, edgecolor='white', linewidth=1.2)

for bar, dep in zip(bars, cluster_prices['Depreciation (%)']):
    label = f'{dep:+.1f}%' if dep != 0 else 'Baseline'
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10000,
            label, ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_xlabel('Damage Tier', fontsize=13)
ax.set_ylabel('Median Price (TL)', fontsize=13)
ax.set_title('Median Car Price by Damage Tier\n(Depreciation Relative to Pristine Baseline)', fontsize=14)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

## 7. Validation — `is_fully_original` Distribution per Cluster

Validate the cluster assignments by checking how many cars in each tier are fully original (unmodified). The Pristine cluster should contain the highest proportion of original cars.

> **⚠️ Replace with your actual cluster results.**

In [ ]:
# ⚠️ Replace with your actual cluster results.
if 'is_fully_original' in df.columns:
    orig_dist = df.groupby(['cluster_label', 'is_fully_original']).size().unstack(fill_value=0)
    orig_dist.columns = ['Modified (0)', 'Fully Original (1)']
    orig_pct = orig_dist.div(orig_dist.sum(axis=1), axis=0) * 100

    fig, ax = plt.subplots(figsize=(10, 6))
    orig_pct.plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'], edgecolor='white')
    ax.set_xlabel('Damage Cluster', fontsize=13)
    ax.set_ylabel('Percentage of Cars (%)', fontsize=13)
    ax.set_title("Validation: Distribution of 'is_fully_original' per Cluster\n(Pristine cluster should contain most original cars)", fontsize=13)
    ax.legend(title='Originality')
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()
else:
    print("Column 'is_fully_original' not found. Skipping validation plot.")

## 8. Visualization — Damage Feature Heatmap

Show the average value of each damage feature per cluster as a heatmap. Use this to verify that your cluster label assignments (Pristine, Light Wear, etc.) make intuitive sense — the Pristine row should be uniformly low.

> **⚠️ Replace with your actual cluster results.**

In [ ]:
# ⚠️ Replace with your actual cluster results.
heatmap_data = df.groupby('cluster_label')[features].mean()
order_existing_for_heatmap = [o for o in order if o in heatmap_data.index]
heatmap_data = heatmap_data.loc[order_existing_for_heatmap]

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Average Value'})
ax.set_title('Average Damage Feature Value per Cluster', fontsize=15)
ax.set_xlabel('Damage Feature', fontsize=12)
ax.set_ylabel('Cluster', fontsize=12)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 9. Visualization — PCA 2D Cluster Scatter Plot

Project the high-dimensional damage features onto 2 principal components and plot the clusters. Well-separated clusters in PCA space indicate the model has found meaningful groupings.

> **⚠️ Replace with your actual cluster results.**

In [ ]:
# ⚠️ Replace with your actual cluster results.
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['Cluster'] = df['cluster_label'].values

palette = {'Pristine': '#2ecc71', 'Light Wear': '#f1c40f',
           'Moderate Damage': '#e67e22', 'Heavy Damage': '#e74c3c'}

fig, ax = plt.subplots(figsize=(10, 7))
for label, group in pca_df.groupby('Cluster'):
    color = palette.get(label, 'gray')
    ax.scatter(group['PC1'], group['PC2'], label=label, alpha=0.5,
               s=30, c=color, edgecolors='none')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
ax.set_title('K-Means Clusters Visualised in PCA Space', fontsize=15)
ax.legend(title='Damage Tier')
plt.tight_layout()
plt.show()
print(f"Total variance explained by 2 PCs: {sum(pca.explained_variance_ratio_)*100:.1f}%")

## ⚠️ If Your Model Underperforms

If your model produces poor clustering (e.g., clusters are mixed in PCA space, or the pristine cluster contains many damaged cars), **do not discard your results**.

- Keep all outputs as-is.
- In your presentation, document exactly what you observe.
- Write a short hypothesis: Why might the model have failed? (e.g., *"The damage features may not have enough variance to separate clusters distinctly because most cars in the dataset have low damage scores."*)